<a href="https://colab.research.google.com/github/Morais9/etl-weather-azure/blob/main/etl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Open-Meteo API Endpoint Reference

This notebook interacts with the Open-Meteo API to fetch weather data. The base URL and example parameters are as follows:

`https://api.open-meteo.com/v1/forecast?latitude=-23.55&longitude=-46.63&daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=America/Sao_Paulo&past_days=7`

- `latitude=-23.55`: Latitude for São Paulo
- `longitude=-46.63`: Longitude for São Paulo
- `daily=temperature_2m_max,temperature_2m_min,precipitation_sum`: Specifies the daily weather variables to retrieve.
- `timezone=America/Sao_Paulo`: Sets the timezone for the data.
- `past_days=7`: Requests 7 days of historical data.

In [ ]:
# =================================================================
# 1. INSTALAÇÃO DE DEPENDÊNCIAS
# =================================================================
# Forçar reinstalação do pandas para evitar 'partially initialized module' errors
!pip uninstall pandas -y --quiet
!pip install pandas==2.2.2 --quiet

!pip install pyodbc sqlalchemy requests --quiet

# Instalação do Driver ODBC 18 para SQL Server (essencial para Azure)
!curl https://packages.microsoft.com/keys/microsoft.asc | apt-key add - > /dev/null 2>&1
!add-apt-repository "$(curl https://packages.microsoft.com/config/ubuntu/$(lsb_release -rs)/prod.list)" -y > /dev/null 2>&1
!apt-get update > /dev/null 2>&1
!ACCEPT_EULA=Y apt-get install -y msodbcsql18 > /dev/null 2>&1

# =================================================================
# 2. IMPORTS E CONFIGURAÇÕES
# =================================================================
import requests
import pandas as pd
import urllib
from datetime import datetime
from sqlalchemy import create_engine, text, event
from google.colab import userdata

CITIES = {
    "Rio Branco": {"lat": -9.97499, "lon": -67.8243},          # Acre
    "Maceió": {"lat": -9.66599, "lon": -35.7350},              # Alagoas
    "Macapá": {"lat": 0.03493, "lon": -51.0694},               # Amapá
    "Manaus": {"lat": -3.11903, "lon": -60.0217},              # Amazonas
    "Salvador": {"lat": -12.9714, "lon": -38.5014},            # Bahia
    "Fortaleza": {"lat": -3.71722, "lon": -38.5433},           # Ceará
    "Brasília": {"lat": -15.7797, "lon": -47.9297},            # Distrito Federal
    "Vitória": {"lat": -20.3155, "lon": -40.3128},             # Espírito Santo
    "Goiânia": {"lat": -16.6864, "lon": -49.2643},             # Goiás
    "São Luís": {"lat": -2.5307, "lon": -44.3068},             # Maranhão
    "Cuiabá": {"lat": -15.6014, "lon": -56.0979},              # Mato Grosso
    "Campo Grande": {"lat": -20.4697, "lon": -54.6201},        # Mato Grosso do Sul
    "Belo Horizonte": {"lat": -19.9167, "lon": -43.9345},      # Minas Gerais
    "Belém": {"lat": -1.4558, "lon": -48.4902},                # Pará
    "João Pessoa": {"lat": -7.11532, "lon": -34.8610},         # Paraíba
    "Curitiba": {"lat": -25.4284, "lon": -49.2733},            # Paraná
    "Recife": {"lat": -8.04756, "lon": -34.8770},              # Pernambuco
    "Teresina": {"lat": -5.08921, "lon": -42.8016},            # Piauí
    "Rio de Janeiro": {"lat": -22.9068, "lon": -43.1729},      # Rio de Janeiro
    "Natal": {"lat": -5.79448, "lon": -35.2110},               # Rio Grande do Norte
    "Porto Alegre": {"lat": -30.0346, "lon": -51.2177},        # Rio Grande do Sul
    "Porto Velho": {"lat": -8.76077, "lon": -63.8999},         # Rondônia
    "Boa Vista": {"lat": 2.81972, "lon": -60.6733},            # Roraima
    "Florianópolis": {"lat": -27.5954, "lon": -48.5480},       # Santa Catarina
    "São Paulo": {"lat": -23.5505, "lon": -46.6333},           # São Paulo
    "Aracaju": {"lat": -10.9472, "lon": -37.0731},             # Sergipe
    "Palmas": {"lat": -10.1840, "lon": -48.3336}               # Tocantins
}

# =================================================================
# 3. EXTRAÇÃO (EXTRACT)
# =================================================================
def extract_weather(city_name, lat, lon, past_days=7):
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat,
        'longitude': lon,
        'daily': ['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max'],
        'timezone': 'America/Sao_Paulo',
        'past_days': past_days
    }
    response = requests.get(url, params=params)
    if response.status_code != 200: return None

    data = response.json()['daily']
    df = pd.DataFrame(data)
    df['city'] = city_name
    df['extracted_at'] = datetime.now()
    return df

all_dfs = [extract_weather(name, c['lat'], c['lon']) for name, c in CITIES.items()]
raw_df = pd.concat([d for d in all_dfs if d is not None], ignore_index=True)

# =================================================================
# 4. TRANSFORMAÇÃO (TRANSFORM)
# =================================================================
df_t = raw_df.rename(columns={
    'time': 'data',
    'temperature_2m_max': 'temp_max_c',
    'temperature_2m_min': 'temp_min_c',
    'precipitation_sum': 'precipitacao_mm',
    'windspeed_10m_max': 'vento_max_kmh',
    'city': 'cidade',
    'extracted_at': 'extraido_em'
})

# Tipagem e Tratamento de Nulos
df_t['data'] = pd.to_datetime(df_t['data'])
df_t['precipitacao_mm'] = df_t['precipitacao_mm'].fillna(0)
df_t['temp_max_c'] = df_t.groupby('cidade')['temp_max_c'].transform(lambda x: x.fillna(x.mean()))

# Novas Colunas (Features)
df_t['amplitude_termica'] = (df_t['temp_max_c'] - df_t['temp_min_c']).round(2)
df_t['temp_media_c'] = ((df_t['temp_max_c'] + df_t['temp_min_c']) / 2).round(1)

def classify_day(row):
    if row['precipitacao_mm'] > 10: return 'Chuvoso'
    if row['temp_max_c'] >= 30: return 'Quente'
    if row['temp_max_c'] < 18: return 'Frio'
    return 'Agradavel'

df_t['classificacao_dia'] = df_t.apply(classify_day, axis=1)
df_t['ano'] = df_t['data'].dt.year
df_t['mes'] = df_t['data'].dt.month
df_t['dia_semana'] = df_t['data'].dt.day_name()

# =================================================================
# 5. CARGA OTIMIZADA (LOAD)
# =================================================================
# Credenciais
server = userdata.get('AZURE_SERVER')
database = userdata.get('AZURE_DB')
username = userdata.get('AZURE_USER')
password = userdata.get('AZURE_PASSWORD')

conn_str = (
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={server};DATABASE={database};UID={username};PWD={password};'
    'Encrypt=yes;TrustServerCertificate=no;'
)

params = urllib.parse.quote_plus(conn_str)
engine = create_engine(f'mssql+pyodbc:///?odbc_connect={params}', fast_executemany=True)

# Evento para garantir o uso do fast_executemany (Otimização de Performance)
@event.listens_for(engine, "before_cursor_execute")
def receive_before_cursor_execute(conn, cursor, statement, parameters, context, executemany):
    if executemany:
        cursor.fast_executemany = True


# Executando a carga
df_t.to_sql(
    name='weather_daily',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=1000
)
# Converter datetime para date
dias_pt = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df_t['dia_semana'] = (
    df_t['data']
    .dt.day_name()
    .map(dias_pt)
)

# Executando a carga
print(f"Enviando {len(df_t)} registros para o Azure SQL...")
print("Carga finalizada com sucesso!")

# =================================================================
# 6. VALIDAÇÃO
# =================================================================
with engine.connect() as conn:
    query = "SELECT TOP 5 cidade, data, temp_media_c, classificacao_dia FROM weather_daily ORDER BY extraido_em DESC"
    check = pd.read_sql(query, conn)
    print("\nÚltimos dados inseridos:")
    print(check)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   975  100   975    0     0   5377      0 --:--:-- --:--:-- --:--:--  5386
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    89  100    89    0     0    578      0 --:--:-- --:--:-- --:--:--   581
Enviando 378 registros para o Azure SQL...
Carga finalizada com sucesso!

Últimos dados inseridos:
   cidade       data  temp_media_c classificacao_dia
0  Palmas 2026-05-06          27.8            Quente
1  Palmas 2026-05-07          27.5            Quente
2  Palmas 2026-05-08          28.4            Quente
3  Palmas 2026-05-09          27.8            Quente
4  Palmas 2026-05-10          28.4            Quente
